# BASE DE DATOS

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

In [ ]:
ruta_train = '../data/fraudTrain.csv'
ruta_test = '../data/fraudTest.csv'

train = pd.read_csv(ruta_train)
test = pd.read_csv(ruta_test)

datos_totales = pd.concat([train, test], axis=0, ignore_index=True)
datos_totales['trans_date_trans_time'] = pd.to_datetime(datos_totales['trans_date_trans_time'])
datos_totales = datos_totales.sort_values(by='trans_date_trans_time').reset_index(drop=True)

if 'Unnamed: 0' in datos_totales.columns:
    datos_totales = datos_totales.drop(columns=['Unnamed: 0'])

print(f"Dataset cargado con {len(datos_totales):,} registros.")
datos_totales.head()

In [ ]:
datos_totales.info()

# Asignando categorías

In [ ]:
cols_to_drop = ['trans_num', 'cc_num', 'first', 'last', 'street', 'unix_time', 'gender', 'dob']
datos_totales = datos_totales.drop(columns=cols_to_drop, errors='ignore')

cols_categoricas = ['merchant', 'category', 'city', 'state', 'job']
for col in cols_categoricas:
    datos_totales[col] = datos_totales[col].astype('category')

datos_totales['amt'] = datos_totales['amt'].astype('float64')
datos_totales['city_pop'] = datos_totales['city_pop'].astype('int64')
datos_totales['is_fraud'] = datos_totales['is_fraud'].astype('int64')

datos_totales.head()

In [ ]:
datos_totales.tail()

# Ordenando fraude por Estados

In [ ]:
verificacion_estados = datos_totales.groupby('state')['is_fraud'].agg(
    Total_Transacciones='count',
    Cantidad_Fraudes='sum'
).reset_index()

verificacion_estados.rename(columns={'state': 'Estado'}, inplace=True)
verificacion_estados['%_Fraude'] = (verificacion_estados['Cantidad_Fraudes'] / verificacion_estados['Total_Transacciones']) * 100
verificacion_estados = verificacion_estados.sort_values(by='Cantidad_Fraudes', ascending=False)

print("--- Estados con Mayor Cantidad de Fraude ---")
display(verificacion_estados.head(15))

# Mapa de calor por propensión de fraude

In [ ]:
import plotly.express as px

riesgo_estado = datos_totales.groupby('state')['is_fraud'].mean().reset_index()
riesgo_estado['is_fraud_pct'] = riesgo_estado['is_fraud'] * 100

fig = px.choropleth(
    riesgo_estado,
    locations='state',
    locationmode="USA-states",
    color='is_fraud_pct',
    scope="usa",
    color_continuous_scale=[
        [0.0, "white"],
        [0.4, "yellow"],
        [1.0, "red"]
    ],
    range_color=[0.3, 0.7],
    labels={'is_fraud_pct': '% de Fraude'},
    title='<b>Riesgo Relativo de Fraude por Estado</b>'
)

fig.update_layout(
    geo_scope='usa',
    margin={"r":0,"t":50,"l":0,"b":0}
)

fig.show()

# Feature Engineering

## Calculando distancia y eliminando variables

In [ ]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c

datos_totales['distancia_km'] = haversine(
    datos_totales['lat'], datos_totales['long'],
    datos_totales['merch_lat'], datos_totales['merch_long']
)

cols_sobrantes = ['lat', 'long', 'merch_lat', 'merch_long', 'city', 'zip']
datos_totales = datos_totales.drop(columns=cols_sobrantes, errors='ignore')

datos_totales.head()

In [ ]:
datos_totales.info()

## Cardinalidad de todas las variables categóricas

In [ ]:
columnas_categoricas = ['merchant', 'category', 'state', 'job']

In [ ]:
for col in columnas_categoricas:
    cardinalidad = datos_totales[col].nunique()
    print(f"Variable '{col}': {cardinalidad} categorías únicas")

## Disminuyendo la cardinalidad de job

In [ ]:
import pandas as pd

categorias_dict = {
    'Healthcare': [
        'Psychologist, counselling', 'Dance movement psychotherapist', 'Pathologist', 'Radiographer, diagnostic',
        'Therapist, occupational', 'Physiotherapist', 'Forensic psychologist', 'Optician, dispensing',
        'Psychologist, forensic', 'Clinical biochemist', 'Paediatric nurse', 'Child psychotherapist',
        'Paramedic', 'Audiological scientist', 'Scientist, audiological', 'Surgeon', 'Therapist, horticultural',
        'Health visitor', 'Medical secretary', 'Diagnostic radiographer', 'Medical physicist', 'Cytogeneticist',
        'Counselling psychologist', 'Chiropodist', 'Psychiatric nurse', 'Embryologist, clinical', 'Immunologist',
        'Health physicist', 'Occupational psychologist', 'Psychologist, sport and exercise', 'Doctor, hospital',
        'Phytotherapist', 'Pharmacologist', 'Horticultural therapist', 'Psychotherapist, child', 'Toxicologist',
        'Podiatrist', 'Mental health nurse', "Nurse, children's", 'Physiological scientist', 'Health and safety adviser',
        'Community pharmacist', 'Art therapist', 'Optometrist', 'Psychotherapist', 'Pharmacist, community',
        'Exercise physiologist', 'Music therapist', 'Acupuncturist', 'Hospital doctor', 'Scientist, physiological',
        'Biochemist, clinical', 'General practice doctor', 'Doctor, general practice', 'Occupational therapist',
        'Neurosurgeon', 'Orthoptist', 'Therapist, drama', 'Therapist, music', 'Dispensing optician',
        'Clinical psychologist', 'Nurse, mental health', 'Pharmacist, hospital', 'Health promotion specialist',
        'Psychiatrist', 'Radiographer, therapeutic', 'Herbalist', 'Osteopath', 'Hospital pharmacist',
        'Nutritional therapist', 'Scientist, research (medical)', 'Psychologist, clinical', 'Medical technical officer',
        'Clinical cytogeneticist', 'Homeopath', 'Veterinary surgeon',
        'Ambulance person', 'Counsellor', 'Therapist, sports', 'Clinical research associate',
        'Learning disability nurse', 'Sport and exercise psychologist', 'Research scientist (medical)', 'Oncologist'
    ],
    'Education': [
        'Special educational needs teacher', 'IT trainer', 'Education officer, museum',
        'Armed forces training and education officer', 'Higher education careers adviser',
        'English as a second language teacher', 'Administrator, education', 'Educational psychologist',
        'Teacher, English as a foreign language', 'Teacher, early years/pre', 'Primary school teacher',
        'Secondary school teacher', 'Librarian, academic', 'Further education lecturer', 'Teacher, secondary school',
        'Museum education officer', 'Teacher, special educational needs', 'Early years teacher',
        'Lecturer, further education', 'Teacher, primary school', 'Professor Emeritus', 'Community education officer',
        'Associate Professor', 'Learning mentor', 'Education administrator', 'Music tutor', 'Lecturer, higher education',
        'Teaching laboratory technician', 'English as a foreign language teacher', 'Academic librarian',
        'Teacher, adult education', 'TEFL teacher',
        'Librarian, public', 'Education officer, community', 'Careers information officer', 'Public librarian',
        'Outdoor activities/education manager', 'Environmental education officer', 'Careers adviser'
    ],
    'Tech & Engineering': [
        'Systems developer', 'Engineer, land', 'Systems analyst', 'Naval architect', 'Energy engineer',
        'Network engineer', 'Applications developer', 'Engineer, mining', 'Electrical engineer',
        'Engineer, technical sales', 'Engineer, electronics', 'Water engineer', 'Engineer, broadcasting (operations)',
        'Engineer, biomedical', 'Mining engineer', 'Engineer, communications', 'Materials engineer',
        'Engineer, structural', 'Structural engineer', 'Database administrator', 'Mechanical engineer',
        'Programmer, multimedia', 'Multimedia programmer', 'Electronics engineer', 'Chemical engineer',
        'Engineer, building services', 'Architectural technologist', 'Chief Technology Officer',
        'Control and instrumentation engineer', 'IT professional', 'Engineer, control and instrumentation',
        'Engineer, maintenance', 'Engineer, production', 'Manufacturing engineer', 'Production engineer',
        'Engineer, manufacturing', 'Engineer, drilling', 'Engineer, petroleum', 'Civil engineer, contracting',
        'Biomedical engineer', 'Building services engineer', 'Maintenance engineer', 'Site engineer',
        'Manufacturing systems engineer', 'Petroleum engineer', 'Communications engineer', 'Drilling engineer',
        'Data scientist', 'Engineer, civil (contracting)', 'IT consultant', 'Aeronautical engineer',
        'Engineer, aeronautical', 'Engineer, civil (consulting)', 'Engineer, materials', 'Broadcast engineer',
        'Engineer, site', 'Contracting civil engineer', 'Software engineer', 'Engineer, water',
        'Telecommunications researcher',
        'Architect', 'Programmer, applications', 'Engineer, agricultural', 'Engineer, automotive',
        'Statistician', 'Energy manager'
    ],
    'Business & Finance': [
        'Public affairs consultant', 'Corporate investment banker', 'Senior tax professional/tax inspector',
        'Economist', 'Purchasing manager', 'Financial adviser', 'Financial trader', 'Accounting technician',
        'Pensions consultant', 'Air broker', 'Advertising account executive', 'Advertising account planner',
        'Investment analyst', 'Pension scheme manager', 'Chief Financial Officer', 'Retail banker',
        'Sales executive', 'Insurance underwriter', 'Retail buyer', 'Equities trader', 'Risk analyst',
        'Logistics and distribution manager', 'Accountant, chartered public finance', 'Buyer, industrial',
        'Comptroller', 'Merchandiser, retail', 'Accountant, chartered certified', 'Chartered public finance accountant',
        'Chief Executive Officer', 'Chief Strategy Officer', 'Chief Operating Officer', 'Marketing executive',
        'Tax inspector', 'Chief Marketing Officer', 'Chartered accountant', 'Buyer, retail', 'Insurance broker',
        'Tax adviser', 'Management consultant', 'Investment banker, corporate', 'Company secretary', 'Media buyer',
        'Investment banker, operational', 'Industrial buyer', 'Accountant, chartered', 'Ship broker', 'Personnel officer',
        'Trade mark attorney', 'Operational researcher', 'Market researcher', 'Social researcher',
        'Social research officer, government', 'Records manager', 'Secretary/administrator', 'Public relations officer',
        'Information systems manager', 'Information officer','Sales professional, IT',
        'Human resources officer', 'Dealer', 'Medical sales representative', 'Training and development officer',
        'Administrator', 'Futures trader', 'Chief of Staff', 'Production manager',
        'Sales promotion account executive', 'Operational investment banker'
    ],
    'Arts & Media': [
        'Designer, multimedia', 'Programme researcher, broadcasting/film/video', 'Designer, furniture', 'Fine artist',
        'Video editor', 'Television camera operator', 'Designer, jewellery', 'Film/video editor',
        'Editor, magazine features', 'Broadcast presenter', 'Producer, radio', 'Theatre director',
        'Television production assistant', 'Exhibition designer', 'Designer, ceramics/pottery', 'Editor, film/video',
        'Camera operator', 'Copywriter, advertising', 'Designer, interior/spatial', 'Production assistant, radio',
        'Jewellery designer', 'Magazine features editor', 'Production assistant, television', 'Illustrator',
        'Designer, industrial/product', 'Writer', 'Special effects artist', 'Radio broadcast assistant',
        'Industrial/product designer', 'Ceramics designer', 'Animator', 'Arts development officer', 'Furniture designer',
        'Editor, commissioning', 'Private music teacher', 'Public relations account executive', 'Musician',
        'Therapist, art', 'Designer, exhibition/display', 'Web designer', 'Press photographer', 'Visual merchandiser',
        'Set designer', 'Television/film/video producer', 'Magazine journalist', 'Textile designer',
        'Glass blower/designer', 'Advertising copywriter', 'Artist', 'Media planner', 'Producer, television/film/video',
        'Broadcast journalist', 'Dancer', 'Designer, television/film set', 'Product designer',
        'Conservator, museum/gallery', 'Museum/gallery exhibitions officer', 'Exhibitions officer, museum/gallery',
        'Sub', 'Make', 'Copy',
        'Curator', 'Interpreter', 'Television floor manager', 'Journalist, newspaper', 'Community arts worker',
        'Radio producer', 'Commissioning editor', 'Press sub', 'Gaffer', 'Theatre manager',
        'Interior and spatial designer', 'Museum/gallery conservator', 'Presenter, broadcasting', 'Designer, textile',
        'Stage manager', 'Art gallery manager', 'Administrator, arts'
    ],
    'Trades & Manual Labor': [
        'Arboriculturist', 'Surveyor, minerals', 'Tree surgeon', 'Freight forwarder', 'Land/geomatics surveyor',
        'Building control surveyor', 'Commercial/residential surveyor', 'Mudlogger', 'Cartographer', 'Contractor',
        'Chartered loss adjuster', 'Building surveyor', 'Minerals surveyor', 'Surveyor, mining', 'Quantity surveyor',
        'Loss adjuster, chartered', 'Pilot, airline', 'Surveyor, land/geomatics', 'Quarry manager',
        'Planning and development surveyor', 'Surveyor, rural practice', 'Insurance risk surveyor',
        'Rural practice surveyor', 'Farm manager', 'Garment/textile technologist', 'Furniture conservator/restorer',
        'Surveyor, hydrographic', 'Airline pilot', 'Technical brewer', 'Land',
        'Transport planner', 'Clothing/textile technologist', 'Hydrographic surveyor', 'Conservator, furniture'
    ],
    'Public Sector & Law': [
        'Patent attorney', 'Probation officer', 'Police officer', 'Research officer, trade union',
        'Research officer, political party', 'Trading standards officer', 'Solicitor, Scotland',
        'Claims inspector/assessor', 'Historic buildings inspector/conservation officer', 'Fisheries officer',
        'Chartered legal executive (England and Wales)', 'Archivist', 'Lexicographer', 'Immigration officer',
        'Barrister', 'Administrator, local government', 'Prison officer', 'Local government officer',
        "Barrister's clerk", "Politician's assistant", 'Insurance claims handler', 'Race relations officer',
        'Advice worker', 'Warden/ranger', 'Equality and diversity officer', 'Town planner', 'Firefighter',
        'Licensed conveyancer', 'Emergency planning/management officer', 'Lawyer', 'Solicitor', 'Legal secretary',
        'Civil Service fast streamer', 'Civil Service administrator', 'Armed forces logistics/support/administrative officer',
        'Armed forces technical officer', 'Administrator, charities/voluntary organisations', 'Charity officer',
        'Charity fundraiser', 'Development worker, community', 'Aid worker',
        'Intelligence analyst', 'Regulatory affairs officer', 'Community development worker',
        'Development worker, international aid'
    ],
    'Science & Nature': [
        'Nature conservation officer', 'Geochemist', 'Scientist, research (maths)', 'Physicist, medical',
        'Amenity horticulturist', 'Science writer', 'Product/process development scientist', 'Geologist, engineering',
        'Research scientist (physical sciences)', 'Operations geologist', 'Agricultural consultant',
        'Waste management officer', 'Environmental consultant', 'Water quality scientist', 'Animal technologist',
        'Occupational hygienist', 'Landscape architect', 'Plant breeder/geneticist', 'Field seismologist',
        'Metallurgist', 'Oceanographer', 'Colour technologist', 'Geoscientist', 'Environmental health practitioner',
        'Chemist, analytical', 'Animal nutritionist', 'Soil scientist', 'Herpetologist', 'Environmental manager',
        'Horticultural consultant', 'Geophysicist/field seismologist', 'Hydrogeologist', 'Geneticist, molecular',
        'Ecologist', 'Horticulturist, commercial', 'Conservation officer, historic buildings',
        'Scientist, clinical (histocompatibility and immunogenetics)', 'Analytical chemist', 'Forest/woodland manager',
        'Engineering geologist', 'Wellsite geologist', 'Geologist, wellsite',
        'Hydrologist', 'Commercial horticulturist', 'Archaeologist', 'Scientist, marine',
        'Research scientist (life sciences)', 'Scientist, research (physical sciences)', 'Scientist, biomedical',
        'Scientific laboratory technician', 'Biomedical scientist', 'Field trials officer', 'Seismic interpreter',
        'Research scientist (maths)'
    ],
    'Service & Retail': [
        'Event organiser', 'Leisure centre manager', 'Call centre manager', 'Tourism officer',
        'Tourist information centre manager', 'Location manager', 'Health service manager', 'Retail merchandiser',
        'Bookseller', 'Facilities manager', 'Public house manager', 'Volunteer coordinator', 'Product manager',
        'Travel agency manager', 'Theme park manager', 'Heritage manager', 'Retail manager', 'Barista', 'Hotel manager',
        'Fitness centre manager', 'Estate manager/land agent', 'Catering manager', 'Warehouse manager',
        'Air cabin crew', 'Cabin crew', 'Restaurant manager, fast food', 'Tour manager', 'Customer service',
        'Air traffic controller',
        'Sports development officer', 'Sports administrator'
    ]
}

mapeo_exacto = {}
for categoria, lista_trabajos in categorias_dict.items():
    for trabajo in lista_trabajos:
        mapeo_exacto[trabajo] = categoria

datos_totales['job_grouped'] = datos_totales['job'].map(mapeo_exacto).fillna('NO_Mapeado')

datos_totales['job_grouped'] = datos_totales['job_grouped'].astype('category')
datos_totales = datos_totales.drop(columns=['job'], errors='ignore')

conteo_areas = datos_totales['job_grouped'].value_counts()
total_inicial = len(datos_totales)
total_categorizado = conteo_areas.sum()

for area, cantidad in conteo_areas.items():
    print(f"{area}: {cantidad:,} registros")

print("\n--- AUDITORÍA MATEMÁTICA ---")
print(f"Total de registros iniciales en la base:   {total_inicial:,}")
print(f"Total de registros categorizados en áreas: {total_categorizado:,}")

if total_inicial == total_categorizado and 'NO_Mapeado' not in conteo_areas:
    print("\n¡Datos completos y perfectos!")
else:
    print("\nDatos incompletos o con desajustes en el mapeo")

In [ ]:
rebeldes_finales = datos_totales[datos_totales['job_grouped'] == 'NO_Mapeado']['job_grouped'].unique().tolist()

print(f"Cantidad de categorías no mapeadas: {len(rebeldes_finales)}\n")
if len(rebeldes_finales) > 0:
    print(", ".join(str(x) for x in rebeldes_finales))
else:
    print("No quedan profesiones sueltas")

## Cardinalidad de las subcategorías de Category, State y job_group

In [ ]:
import pandas as pd

columnas_categoricas = ['category', 'state', 'job_grouped']

for col in columnas_categoricas:
    if col in datos_totales.columns:
        print(f"Variable: {col.upper()} con un total de subcategorías: {datos_totales[col].nunique()}")
        
        conteo = datos_totales[col].value_counts()
        porcentaje = (conteo / conteo.sum()) * 100
        
        tabla_atributos = pd.DataFrame({
            'Atributo': conteo.index,
            'Frecuencia (N)': conteo.values,
            'Porcentaje (%)': porcentaje.values.round(4)
        })
        
        display(tabla_atributos)
        print("-" * 60, "\n")
    else:
        print(f"La columna {col} no se encuentra en el dataset actual.")

In [ ]:
datos_totales.head()

In [ ]:
datos_totales.info()

In [ ]:
datos_totales = datos_totales.drop(columns=['job', 'merchant'], errors='ignore')
print(datos_totales.columns)

In [ ]:
datos_totales.head()

## Tablas de contingencia y pruebas chi cuadrado de Pearson

Prueba de Hipótesis: Chi-Cuadrado de Pearson

* **Hipótesis Nula ($H_0$):** La variable categórica y la ocurrencia de fraude (`is_fraud`) son **independientes**. No existe una relación sistemática entre la variable categórica y el fraude.
* **Hipótesis Alternativa ($H_1$):** Existe una **asociación o dependencia** entre la variable categórica y el fraude. El comportamiento del fraude varía según la variable categórica.

2. Criterio de Decisión
* Si **p-valor < 0.05**: Rechazamos $H_0$. La relación es estadísticamente significativa.
* Si **p-valor ≥ 0.05**: No hay evidencia suficiente para rechazar $H_0$.

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

variables_analisis = ['job_grouped', 'category', 'state']

print("=== ANÁLISIS DE DEPENDENCIAS (CHI-CUADRADO) ===\n")

for col in variables_analisis:
    if col in datos_totales.columns:
        print(f"TEST CHI-CUADRADO PARA: {col.upper()}")
        print("-" * 40)
        
        tabla_contingencia = pd.crosstab(datos_totales[col], datos_totales['is_fraud'])
        
        tabla_porcentual = pd.crosstab(datos_totales[col], datos_totales['is_fraud'], normalize='index') * 100
        print("Distribución porcentual (Ordenada por tasa de fraude):")
        print(tabla_porcentual.sort_values(by=1, ascending=False))
        print("\nPrueba Estadística:")
        
        chi2, p, dof, expected = chi2_contingency(tabla_contingencia)
        
        print(f"Estadístico Chi-cuadrado: {chi2:.4f}")
        print(f"P-valor: {p:.10e}")
        print(f"Grados de libertad: {dof}")
        
        if p < 0.05:
            print("Resultado: Existe una relación significativa.")
            print(f"   La variable '{col}' influye directamente en el fraude y no es por azar.\n")
        else:
            print("Resultado: No hay relación significativa.")
            print("   Las diferencias en los porcentajes podrían ser mera casualidad.\n")
            
        print("=" * 60 + "\n")
    else:
        print(f"La columna {col} no se encuentra en el DataFrame actual.\n")

## DUMMIES

In [ ]:
import pandas as pd

variables_dummies = ['category', 'job_grouped', 'state']
prefijos = ['cat', 'job', 'state']
columnas_a_eliminar = []

for var, prefijo in zip(variables_dummies, prefijos):
    if var in datos_totales.columns:
        categoria_base = datos_totales[var].mode()[0]
        columna_dummy_base = f"{prefijo}_{categoria_base}"
        columnas_a_eliminar.append(columna_dummy_base)
        print(f"Variable '{var}': Base más frecuente es '{categoria_base}' -> Se eliminará '{columna_dummy_base}'")

datos_totales = pd.get_dummies(
    datos_totales,
    columns=variables_dummies,
    prefix=prefijos,
    drop_first=False,
    dtype=int
)

datos_totales = datos_totales.drop(columns=columnas_a_eliminar, errors='ignore')

print(f"\nProcesamiento completo. Total de columnas finales: {len(datos_totales.columns)}")

In [ ]:
nulos_por_columna = datos_totales.isnull().sum()
nulos_totales = nulos_por_columna.sum()

print(f"Total de valores nulos en la base: {nulos_totales}")

if nulos_totales > 0:
    print("\nDetalle de nulos por columna:")
    print(nulos_por_columna[nulos_por_columna > 0])
else:
    print("El dataset está 100%")

In [ ]:
datos_totales.info()

## Caracterización de las variables numéricas

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

vars_plot = ['amt', 'city_pop', 'distancia_km']

fig = make_subplots(
    rows=2, cols=2, 
    subplot_titles=vars_plot
)

for i, var in enumerate(vars_plot):
    row = (i // 2) + 1
    col = (i % 2) + 1

    fig.add_trace(
        go.Histogram(
            x=datos_totales[var], 
            name=var, 
            marker_color='#636EFA'
        ),
        row=row, col=col
    )

fig.update_layout(
    title_text="<b>Análisis Exploratorio: Distribución de Variables Numéricas</b>",
    template="plotly_white",
    height=800,
    showlegend=False,
    dragmode="zoom", 
    hovermode="x unified" 
)

fig.update_xaxes(fixedrange=False)
fig.update_yaxes(fixedrange=False)

fig.show()

In [ ]:
estadisticos = datos_totales[['amt', 'city_pop', 'distancia_km']].describe().loc[['min', 'max', 'mean', 'std']]
print(estadisticos)

In [ ]:
datos_totales.info()

## Revisión de normalidad de las variables numéricas

Prueba de Normalidad 
Establecemos formalmente la prueba para validar la necesidad de estas transformaciones:
* **$H_0$:** La variable sigue una distribución normal.
* **$H_1$:** La variable NO sigue una distribución normal.

In [ ]:
from scipy.stats import normaltest
from sklearn.preprocessing import RobustScaler, MinMaxScaler
import numpy as np

variables = ['amt', 'city_pop', 'distancia_km']

print("--- Prueba de Normalidad ---")
for var in variables:
    stat, p = normaltest(datos_totales[var])
    print(f"{var.upper()}: p-valor = {p:.10f} -> {'No Normal' if p < 0.05 else 'Normal'}")

## Aplicando logaritmo a amt

In [ ]:
import numpy as np

datos_totales['amt_log'] = np.log1p(datos_totales['amt'])

datos_totales = datos_totales.drop(columns=['amt'], errors='ignore')

print("¡Transformación Log1p aplicada con éxito!")
print(f"Columnas actuales en el dataset: {len(datos_totales.columns)}")

In [ ]:
datos_totales.info()

## SPLIT DE LA BASE DE DATOS

In [ ]:
datos_totales = datos_totales.sort_values(by='trans_date_trans_time').reset_index(drop=True)

total_filas = len(datos_totales)
corte_train = int(total_filas * 0.70)
corte_test = int(total_filas * 0.85)

fecha_corte_train = datos_totales.loc[corte_train, 'trans_date_trans_time']
fecha_corte_test = datos_totales.loc[corte_test, 'trans_date_trans_time']

df_train = datos_totales[datos_totales['trans_date_trans_time'] < fecha_corte_train].copy()
df_test = datos_totales[(datos_totales['trans_date_trans_time'] >= fecha_corte_train) & 
                        (datos_totales['trans_date_trans_time'] < fecha_corte_test)].copy()
df_oot = datos_totales[datos_totales['trans_date_trans_time'] >= fecha_corte_test].copy()

print(f"Partición Temporal Exacta (70/15/15) Completada:")
print(f" -> Train (Primeros 70%):  {df_train.shape[0]} filas ({round(len(df_train)/total_filas*100, 2)}%)")
print(f" -> Test  (Siguientes 15%): {df_test.shape[0]} filas ({round(len(df_test)/total_filas*100, 2)}%)")
print(f" -> OOT   (Últimos 15%):    {df_oot.shape[0]} filas ({round(len(df_oot)/total_filas*100, 2)}%)")

In [ ]:
df_train.info()

## Cambiando nombres a los estados

In [ ]:
mapeo_completo = {
    'amt_log': 'monto_log',
    'city_pop_scaled': 'poblacion_ciudad_escalado',
    'distancia_scaled': 'distancia_km_escalado',
    'tm_hora': 'tiempo_hora',
    'tm_dia_semana': 'tiempo_dia_semana',
    'tm_mes': 'tiempo_mes',
    'is_fraud': 'es_fraude',

    'cat_entertainment': 'cat_entretenimiento', 'cat_food_dining': 'cat_comida_restaurantes',
    'cat_grocery_net': 'cat_supermercado_online', 'cat_grocery_pos': 'cat_supermercado_fisico',
    'cat_health_fitness': 'cat_salud_gimnasio', 'cat_home': 'cat_hogar', 'cat_kids_pets': 'cat_ninos_mascotas',
    'cat_misc_net': 'cat_miscelaneo_online', 'cat_misc_pos': 'cat_miscelaneo_fisico',
    'cat_personal_care': 'cat_cuidado_personal', 'cat_shopping_net': 'cat_compras_online',
    'cat_shopping_pos': 'cat_compras_fisico', 'cat_travel': 'cat_viajes',

    'job_Arts & Media': 'job_Artes_y_Medios', 'job_Business & Finance': 'job_Negocios_y_Finanzas',
    'job_Education': 'job_Educacion', 'job_Public Sector & Law': 'job_Sector_Publico_y_Derecho',
    'job_Science & Nature': 'job_Ciencia_y_Naturaleza', 'job_Service & Retail': 'job_Servicios_y_Comercio',
    'job_Tech & Engineering': 'job_Tecnologia_e_Ingenieria', 'job_Trades & Manual Labor': 'job_Oficios_y_Trabajo_Manual',

    'state_AK': 'estado_Alaska', 'state_AL': 'estado_Alabama', 'state_AR': 'estado_Arkansas', 
    'state_AZ': 'estado_Arizona', 'state_CA': 'estado_California', 'state_CO': 'estado_Colorado', 
    'state_CT': 'estado_Connecticut', 'state_DC': 'estado_Distrito_de_Columbia', 'state_DE': 'estado_Delaware',
    'state_FL': 'estado_Florida', 'state_GA': 'estado_Georgia', 'state_HI': 'estado_Hawai', 
    'state_IA': 'estado_Iowa', 'state_ID': 'estado_Idaho', 'state_IL': 'estado_Illinois', 
    'state_IN': 'estado_Indiana', 'state_KS': 'estado_Kansas', 'state_KY': 'estado_Kentucky', 
    'state_LA': 'estado_Luisiana', 'state_MA': 'estado_Massachusetts', 'state_MD': 'estado_Maryland', 
    'state_ME': 'estado_Maine', 'state_MI': 'estado_Michoacan', 'state_MN': 'estado_Minnesota', 
    'state_MO': 'estado_Misuri', 'state_MS': 'estado_Misisipi', 'state_MT': 'estado_Montana', 
    'state_NC': 'estado_Carolina_del_Norte', 'state_ND': 'estado_Dakota_del_Norte', 'state_NE': 'estado_Nebraska', 
    'state_NH': 'estado_Nuevo_Hampshire', 'state_NJ': 'estado_Nueva_Jersey', 'state_NM': 'estado_Nuevo_Mexico', 
    'state_NV': 'estado_Nevada', 'state_NY': 'estado_Nueva_York', 'state_OH': 'estado_Ohio', 
    'state_OK': 'estado_Oklahoma', 'state_OR': 'estado_Oregon', 'state_PA': 'estado_Pensilvania', 
    'state_RI': 'estado_Rhode_Island', 'state_SC': 'estado_Carolina_del_Sur', 'state_SD': 'estado_Dakota_del_Sur', 
    'state_TN': 'estado_Tennessee', 'state_UT': 'estado_Utah', 'state_VA': 'estado_Virginia', 
    'state_VT': 'estado_Vermont', 'state_WA': 'estado_Washington', 'state_WI': 'estado_Wisconsin', 
    'state_WV': 'estado_Virginia_Occidental', 'state_WY': 'estado_Wyoming'
}

df_train = df_train.rename(columns=mapeo_completo)
df_test = df_test.rename(columns=mapeo_completo)
df_oot = df_oot.rename(columns=mapeo_completo)

print("¡Traducción 100% completada!")
print("\nValidación:")
columnas_interes = ['es_fraude', 'job_Artes_y_Medios', 'estado_California', 'monto_log']
for col in columnas_interes:
    status = "Bien" if col in df_train.columns else "Mal"
    print(f" - {col}: {status}")

In [ ]:
df_train.info()

# REVISANDO MULTICOLINEALIDAD

In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_train_sample_multicolinealidad = df_train.drop(columns=['es_fraude','trans_date_trans_time'], errors='ignore').sample(n=50000, random_state=42)

vif_data = pd.DataFrame()
vif_data["Variable"] = X_train_sample_multicolinealidad.columns

print("Calcular VIF optimizado sobre muestra representativa")
vif_data["VIF"] = [variance_inflation_factor(X_train_sample_multicolinealidad.values, i) for i in range(X_train_sample_multicolinealidad.shape[1])]

vif_data = vif_data.sort_values(by="VIF", ascending=False).reset_index(drop=True)
print("\nTop 15 Variables con Mayor Multicolinealidad (Muestreado):")
print(vif_data.head(15))

In [ ]:
print(vif_data.head(76))

# Usando otro nombre

In [ ]:
dataset_entrenamiento = df_train
dataset_prueba = df_test
dataset_OOT = df_oot

## Revisando otra vez variables

In [ ]:
dataset_entrenamiento.info()

In [ ]:
dataset_prueba.info()

In [ ]:
dataset_OOT.info()

# ESCALANDO Y EVITANDO DATA LEAKAGE

In [ ]:
from sklearn.preprocessing import PowerTransformer

cols_numericas = ['city_pop', 'distancia_km']

pt = PowerTransformer(method='yeo-johnson', standardize=True)

dataset_entrenamiento[cols_numericas] = pt.fit_transform(dataset_entrenamiento[cols_numericas])

display(dataset_entrenamiento[cols_numericas].describe().round(4))

In [ ]:
from sklearn.preprocessing import PowerTransformer

cols_numericas = ['city_pop', 'distancia_km']

pt = PowerTransformer(method='yeo-johnson', standardize=True)

dataset_prueba[cols_numericas] = pt.fit_transform(dataset_prueba[cols_numericas])

import joblib

joblib.dump(pt, '../data/power_transformer.pkl')

display(dataset_prueba[cols_numericas].describe().round(4))

In [ ]:
from sklearn.preprocessing import PowerTransformer

cols_numericas = ['city_pop', 'distancia_km']

pt = PowerTransformer(method='yeo-johnson', standardize=True)

dataset_OOT[cols_numericas] = pt.fit_transform(dataset_OOT[cols_numericas])

display(dataset_OOT[cols_numericas].describe().round(4))

In [ ]:
dataset_entrenamiento = dataset_entrenamiento.drop(columns=['trans_date_trans_time'], errors='ignore')
dataset_prueba = dataset_prueba.drop(columns=['trans_date_trans_time'], errors='ignore')
dataset_OOT = dataset_OOT.drop(columns=['trans_date_trans_time'], errors='ignore')

In [ ]:
dataset_entrenamiento.info()

In [ ]:
dataset_entrenamiento.head()

# Revisión de desbalanceo

In [ ]:
proporcion_fraude = df_train['es_fraude'].value_counts(normalize=True) * 100
proporcion_fraude

In [ ]:
dataset_entrenamiento.to_csv('../data/train_clean.csv', index=False)
dataset_prueba.to_csv('../data/test_clean.csv', index=False)
dataset_OOT.to_csv('../data/oot_clean.csv', index=False)